# 04 - Explainability: Grad-CAM on the validation set

**Goal:** check which regions of the retinal image the model uses for its predictions: lesions and retinal structures, or artifacts (black corners, camera notch, borders)?

**Model:** `efficientnet_b0_v2_best.pth` (trained on the cleaned data). **Images:** validation set only (701 images).

**Important:** Grad-CAM shows where the model looked, not why. At 512 px the heatmap has only 16x16 cells, so it shows regions and not individual lesions. It is a sanity check, not proof of correct reasoning.

## 1. Loading the model and reproducing the validation metrics
We reload the saved weights and recompute the validation metrics. They must match notebook 03 (QWK 0.926, accuracy 0.862); if they do not, the wrong weights were loaded and nothing below can be trusted.

In [2]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn
import matplotlib.pyplot as plt

from src.model import create_model
from src.dataset import get_dataloaders, IMAGENET_MEAN, IMAGENET_STD, DATA
from src.engine import evaluate
from src.gradcam import GradCAM, overlay

device = torch.device("cuda")
model = create_model(pretrained=False).to(device)
model.load_state_dict(torch.load("../models/efficientnet_b0_v2_best.pth", map_location=device))
model.eval()

_, val_loader, _, val_ds = get_dataloaders(batch_size=16, num_workers=0)
m = evaluate(model, val_loader, nn.CrossEntropyLoss(), device)
probs, y = m["probs"], m["y_true"]
pred = probs.argmax(axis=1)
print("QWK:", round(m["qwk"], 3), "| accuracy:", round(m["acc"], 3), "| val images:", len(val_ds))

QWK: 0.926 | accuracy: 0.862 | val images: 701


**Interpretation:** QWK and accuracy match notebook 03, so the correct weights are loaded and the validation order is the same.

## 2. Heatmap helper
`explain(i)` runs Grad-CAM on validation image `i`; `show` draws the original image (top) and the heatmap overlay (bottom). The target layer is `model.bn2`, the last feature map before pooling. Red means a stronger influence on the class score.

In [3]:
gradcam = GradCAM(model, model.bn2)
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
names = ["No DR", "Mild", "Moderate", "Severe", "Prolif."]

def explain(i, use_true=False):
    x, label = val_ds[i]
    cam, p, c = gradcam(x.unsqueeze(0).to(device), label if use_true else None)
    img = (x * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    return img, cam, p, c, label

def show(indices, use_true=False):
    fig, axes = plt.subplots(2, len(indices), figsize=(4 * len(indices), 8), squeeze=False)
    for j, i in enumerate(indices):
        img, cam, p, c, label = explain(int(i), use_true)
        axes[0, j].imshow(img)
        axes[0, j].set_title(f"row {i} | true {names[label]}", fontsize=10)
        axes[1, j].imshow(overlay(img, cam))
        axes[1, j].set_title(f"map for {names[c]} (p={p[c]:.2f})", fontsize=10)
        axes[0, j].axis("off"); axes[1, j].axis("off")
    plt.tight_layout(); plt.show()

## 3. Correct predictions: one random image per grade
Images are chosen at random (fixed seed), not selected for looking good, so the sample is not cherry-picked.

In [ ]:
rng = np.random.default_rng(0)
picks = [int(rng.choice(np.where((y == g) & (pred == g))[0])) for g in range(5)]
show(picks)

**Interpretation:** 

The five images were chosen at random among correct predictions, not selected for looking good.
- In several maps the hottest region sits on structures that could be lesions: the Mild map has one focal spot near the macular region, the strongest Moderate spot coincides with a cluster of small yellowish dots in the lower left, and the Severe map covers the large exudate cluster. These are visual impressions, not clinically validated.
- In 3 of the 5 images (Moderate, Severe, Proliferative), strong heat also lies on the image border: in the black corners or along the top and bottom edges. In the Proliferative image the strongest heat is on those edges rather than on the central lesions.
- The No DR map is diffuse, as expected for "no disease", with a hot spot next to the camera notch.
- The 16x16 maps cannot outline individual lesions.

## 4. Errors: Moderate images predicted as Mild
This confusion causes most missed referable patients (24 of 185 Moderate images). We look at where the model looked, for its predicted class (Mild, top) and for the true class (Moderate, bottom).

In [5]:
idx = np.where((y == 2) & (pred == 1))[0]
print(len(idx), "Moderate images predicted Mild")
sel = rng.choice(idx, 5, replace=False)
show(sel)                  # map for the predicted class (Mild)
show(sel, use_true=True)   # map for the true class (Moderate)

24 Moderate images predicted Mild


**Interpretation:** for 5 randomly chosen Moderate images predicted as Mild, the predicted-class (Mild) probability is 0.34 to 0.73, and the maps show small focal hot spots, in some cases near small yellowish dots. The maps for the true class (Moderate, p = 0.18 to 0.43) largely coincide with the Mild maps in two of the five images, so the model looks at the same places for both classes, and the confusion seems to come from the amount of evidence assigned rather than from looking elsewhere. In two images (rows 41 and 570) the Moderate map places strong heat in the black corners. Five images are anecdotes, not statistics.

## 5. The two referable patients missed at every threshold
Validation rows 9 (true grade 4) and 277 (true grade 2) keep a referable probability below 0.05. Maps for the predicted class and for the true class.

In [6]:
for i in [9, 277]:
    print(f"row {i} | true {names[y[i]]} | predicted {names[pred[i]]} | probabilities {probs[i].round(3)}")
show([9, 277])
show([9, 277], use_true=True)

row 9 | true Prolif. | predicted Mild | probabilities [0.005 0.967 0.021 0.004 0.001]
row 277 | true Moderate | predicted Mild | probabilities [0.004 0.961 0.032 0.002 0.002]


**Interpretation:** these two patients keep a summed referable probability below 0.05, so no threshold recovers them. The maps are for the *true* class, and p is the probability the model gives to that class.
- **Row 277 (true Moderate, p(Moderate) = 0.03):** the hottest region is a dark red spot at the right periphery that could be a hemorrhage. The heat shows that this region raises the Moderate score, not that the model recognized a lesion, and the model still gives Moderate very little probability.
- **Row 9 (true Proliferative, p(Proliferative) ≈ 0.00):** the heat is on the top edge and the bottom-right periphery, and no lesion is identifiable by eye there.
- **Predicted classes:** both patients are predicted Mild with high probability (row 9: 0.967, probabilities 0.005 / 0.967 / 0.021 / 0.004 / 0.001; row 277: 0.961, probabilities 0.004 / 0.961 / 0.032 / 0.002 / 0.002). The model is confidently wrong on these two cases, not borderline.
- These cases were not reviewed by a clinician, so they may be label errors or genuine model failures.

## 6. Does the heatmap fall on the black background?

For every validation image we measure, on the processed image, the black background (pixels with brightness 20 or less), and then:
- the share of the image that is background (16% on average);
- the share of the heatmap mass that falls on the background, compared with the area share (a uniform map would give a ratio of 1);
- whether the single hottest pixel is on the background;
- whether more than 25% of the heat is on the background.

This is approximate: the 16x16 map is upsampled, so some heat bleeds across the retina boundary even for a perfect model, and the ratio is not expected to reach 0.

In [7]:
heat_on_bg, area_bg, peak_on_bg = [], [], []
for i in range(len(val_ds)):
    x, _ = val_ds[i]
    cam, _, _ = gradcam(x.unsqueeze(0).to(device))
    code = val_ds.df.iloc[i]["id_code"]
    gray = cv2.imread(str(DATA / "processed_512" / f"{code}.png"), cv2.IMREAD_GRAYSCALE)
    bg = gray <= 20
    area_bg.append(bg.mean())
    heat_on_bg.append(cam[bg].sum() / (cam.sum() + 1e-8))
    peak_on_bg.append(bg.flat[cam.argmax()])

area_bg, heat_on_bg, peak_on_bg = map(np.array, (area_bg, heat_on_bg, peak_on_bg))

print("Mean share of black area in the image:               ", round(area_bg.mean(), 3))
print("Mean share of heatmap mass on black:                 ", round(heat_on_bg.mean(), 3))
print("Ratio (heat share / area share):                     ", round(heat_on_bg.mean() / area_bg.mean(), 2))
print("Images whose hottest pixel is on the background:     ", round(float(peak_on_bg.mean()), 3))
print("Images with more than 25% of the heat on background: ", round(float((heat_on_bg > 0.25).mean()), 3))

Mean share of black area in the image:                0.162
Mean share of heatmap mass on black:                  0.139
Ratio (heat share / area share):                      0.86
Images whose hottest pixel is on the background:      0.04
Images with more than 25% of the heat on background:  0.116


**Interpretation**
- The black background covers 16.2% of the processed image on average, and 13.9% of the heatmap mass falls on it (ratio 0.86). A uniform map would give a ratio of 1, so this measure is only slightly below the uniform baseline. It does not establish how much the model relies on background pixels: the map is a coarse, upsampled visualization, and low-level heat accumulates over large areas.
- The single hottest pixel falls on the background in 4% of images, so a strong background hot spot is uncommon. In 11.6% of images, more than a quarter of the heat mass falls on the background, so a noticeable minority of maps put substantial heat there. Several individual maps show strong corner or border heat (rows 41, 353 and 570).
- These figures are a signal to look at those cases, not proof of reliance on the background. The masking test in section 7 examines this directly.

## 7. Sensitivity to background pixels (masking test)

We set selected pixels of every validation image to pure black and re-run the model without retraining. Variants: (1) all pixels with brightness 20 or less; (2) everything outside the retina's field of view (FOV, the convex hull of the largest bright region); (3) only the dark pixels inside the field of view. A control run with no edit checks that the pipeline reproduces notebook 03. For referable patients we count how many were newly missed or newly caught and run an exact binomial (McNemar-type) test on those counts. This is exploratory: one validation set, no correction for the three variants.

In [ ]:
grays = [cv2.imread(str(DATA / "processed_512" / f"{val_ds.df.iloc[i]['id_code']}.png"),
                    cv2.IMREAD_GRAYSCALE) for i in range(len(val_ds))]

def fov_mask(gray, thr=20):
    """True inside the retina (convex hull of the largest bright region)."""
    m = (gray > thr).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m)
    if n < 2:
        return np.ones_like(gray, dtype=bool)
    m = (lab == 1 + stats[1:, cv2.CC_STAT_AREA].argmax()).astype(np.uint8)
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    hull = cv2.convexHull(max(cnts, key=cv2.contourArea))
    out = np.zeros_like(m)
    cv2.fillConvexPoly(out, hull, 1)
    return out.astype(bool)

fovs = [fov_mask(g) for g in grays]

# visual check: red line = FOV boundary
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, i in zip(axes.ravel(), np.random.default_rng(1).choice(len(val_ds), 10, replace=False)):
    ax.imshow(grays[i], cmap="gray")
    ax.contour(fovs[i].astype(float), levels=[0.5], colors="r", linewidths=1)
    ax.set_title(f"row {i}", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

**Interpretation:** on 10 random validation images the red contour follows the retinal boundary closely, with a straight cut at the camera notch, so the mask is reliable for separating pixels outside and inside the retina.

In [10]:
from sklearn.metrics import cohen_kappa_score
from scipy.stats import binomtest

mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

is_referable = np.asarray(y) >= 2

def referral_metrics(probs, threshold=0.50):
    p_referable = probs[:, 2:].sum(axis=1)
    pred_referable = p_referable >= threshold
    tp = int(np.sum(pred_referable & is_referable))
    fn = int(np.sum(~pred_referable & is_referable))
    tn = int(np.sum(~pred_referable & ~is_referable))
    fp = int(np.sum(pred_referable & ~is_referable))
    return {
        "threshold": threshold,
        "sensitivity": tp / (tp + fn),
        "specificity": tn / (tn + fp),
        "missed referable": fn,
        "false alarms": fp,
    }

black = (0 - mean) / std

def predict_masked(masks, bs=32):
    """masks[i] = boolean (512, 512) array, True where pixels are set to pure black."""
    out_p = []
    for s in range(0, len(val_ds), bs):
        batch = []
        for i in range(s, min(s + bs, len(val_ds))):
            x, _ = val_ds[i]
            bg = torch.from_numpy(masks[i]).unsqueeze(0)
            batch.append(torch.where(bg, black.expand_as(x), x))
        with torch.enable_grad():        # kept: GradCAM hooks may need it
            out = model(torch.stack(batch).to(device))
        out_p.append(torch.softmax(out.detach(), dim=1).cpu().numpy())
    return np.concatenate(out_p)

none = [np.zeros_like(g, dtype=bool) for g in grays]
variants = {
    "original (control)":        none,
    "old: gray <= 20":           [g <= 20 for g in grays],
    "outside FOV only":          [~f for f in fovs],
    "dark pixels inside FOV":    [f & (g <= 20) for f, g in zip(fovs, grays)],
}
probs_by = {name: predict_masked(m) for name, m in variants.items()}
p0 = probs_by["original (control)"]
print("Changed with no edit vs evaluate():", int((p0.argmax(1) != pred).sum()), "of", len(y))

rows = []
for name, p in probs_by.items():
    pr = p.argmax(1)
    r = referral_metrics(p)
    ra, rb = p0[:, 2:].sum(1) >= 0.5, p[:, 2:].sum(1) >= 0.5
    b = int((is_referable & ra & ~rb).sum())      # newly missed
    c = int((is_referable & ~ra & rb).sum())      # newly caught
    rows.append({
        "variant": name,
        "QWK": cohen_kappa_score(y, pr, weights="quadratic"),
        "acc": (pr == y).mean(),
        "sens": r["sensitivity"], "spec": r["specificity"],
        "missed": r["missed referable"], "false alarms": r["false alarms"],
        "grade changes": int((p0.argmax(1) != pr).sum()),
        "newly missed": b, "newly caught": c,
        "McNemar p": binomtest(b, b + c, 0.5).pvalue if b + c else 1.0,
    })
display(pd.DataFrame(rows).set_index("variant").round(3))

Changed with no edit vs evaluate(): 2 of 701


,QWK,acc,sens,spec,missed,false alarms,grade changes,newly missed,newly caught,McNemar p
variant,,,,,,,,,,
original (control),0.925,0.859,0.920,0.965,22,15,0,0,0,1.000
old: gray <= 20,0.908,0.840,0.880,0.965,33,15,32,12,1,0.003
outside FOV only,0.908,0.837,0.869,0.967,36,14,30,15,1,0.001
dark pixels inside FOV,0.924,0.859,0.905,0.963,26,16,9,4,0,0.125


**Interpretation**
- **Control:** 2 of 701 predictions differ from notebook 03 (full precision here, mixed precision there); QWK 0.925 versus 0.926.
- **Outside the retina:** setting these pixels to pure black lowers QWK from 0.925 to 0.908 and referable sensitivity from 0.920 to 0.869 (36 missed instead of 22), with 30 grade changes. Among referable patients whose decision changed, 15 were newly missed and 1 newly caught (p = 0.001). The "all pixels ≤ 20" variant gives a similar result (12 versus 1, p = 0.003).
- **Dark pixels inside the retina:** a much smaller effect (QWK 0.924, 9 grade changes, 4 versus 0, p = 0.125, not significant), so the effect comes mainly from pixels outside the retina.
- Specificity is unchanged (0.963 to 0.967).
- **Limits:** this shows that the model is sensitive to the values of background pixels, not that it extracts diagnostic information from them. After CLAHE the background is faint texture, not pure black, and pure black is a pattern never seen in training, so the effect may reflect a distribution shift. In both cases, referral decisions depend on pixels that carry no retinal information, which is a risk for data from other cameras. Exploratory: validation set only.

## Summary

| Check | Result |
|---|---|
| Correct predictions (5 random images) | Several hot spots coincide with possible lesions; in 3 of 5 images strong heat also lies on borders or corners |
| Moderate predicted as Mild (5 random images) | Maps for the predicted and the true class largely coincide; the confusion seems to come from the amount of evidence, not from looking elsewhere |
| Two patients missed at every threshold | Both predicted Mild with probability 0.96 to 0.97; one shows heat on a possible hemorrhage, the other shows no identifiable lesion at the hot spot; no clinician review |
| Heat on the black background | 13.9% of the heat mass for 16.2% of the area (ratio 0.86); hottest pixel on the background in 4% of images; more than 25% of the heat on the background in 11.6% |
| Masking test | Blacking out pixels outside the retina lowers referable sensitivity from 0.920 to 0.869 (15 newly missed versus 1 newly caught, p = 0.001); inside the retina the effect is small and not significant |

**Conclusion:** the model is partly sensitive to pixels outside the retina. This does not prove it uses them diagnostically, but it makes its predictions fragile to changes in camera and borders.

**Limitations:** coarse 16x16 maps, qualitative inspection of few images, no lesion annotations, no clinician review, exploratory tests on the validation set.

**Next:** decide whether to remove the background in preprocessing and retrain, then external validation.